In [3]:
import sys
import pandas as pd
import numpy as np
import random
import tensorflow as tf
import plotly.express as px
import plotly.graph_objects as go
import os
os.environ["PYTHONHASHSEED"] = "42"
os.environ["TF_DETERMINISTIC_OPS"] = "1"
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from pathlib import Path

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "src").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.simulation.dgp0 import Tier0Config, simulate_panel
from src.simulation.validation import plot_market_plotly, separation_auc_like
from src.data.feature_eng import feature_eng_syn
from src.simulation.windows.windows import make_windows
from src.data.cartel_data import prepare_cartel_panel

In [4]:
df1 = pd.read_csv("../data/cartel_data/cartel_prices_monthly_combined.csv")
df1.shape

(191, 5)

In [5]:
df2 = pd.read_csv("../data/cartel_data/vitamin_data.csv")
df2.shape

(576, 5)

In [ ]:
cartel_1 = prepare_cartel_panel(df1)
cartel_1.head()

,market_id,date,t,p,S,collusion_flag,unit,price,raw_price
0,citric_acid,1987-01-01,0,4.369448,0,0,cents_per_lb,79.00,79.00
1,citric_acid,1987-02-01,1,4.366278,0,0,cents_per_lb,78.75,78.75
2,citric_acid,1987-03-01,2,4.363353,0,0,cents_per_lb,78.52,78.52
3,citric_acid,1987-04-01,3,4.360037,0,0,cents_per_lb,78.26,78.26
4,citric_acid,1987-05-01,4,4.356837,0,0,cents_per_lb,78.01,78.01


In [16]:
cartel_2 = prepare_cartel_panel(df2)
cartel_2.head()

,market_id,date,t,p,S,collusion_flag,unit,price_index,raw_price
0,beta_carotene,1990-01-01,0,4.248495,2,1,index_Jan1995_100,70.000000,70.000000
1,beta_carotene,1990-02-01,1,4.254543,2,1,index_Jan1995_100,70.424658,70.424658
2,beta_carotene,1990-03-01,2,4.259975,2,1,index_Jan1995_100,70.808219,70.808219
3,beta_carotene,1990-04-01,3,4.265954,2,1,index_Jan1995_100,71.232877,71.232877
4,beta_carotene,1990-05-01,4,4.271707,2,1,index_Jan1995_100,71.643836,71.643836


In [18]:
cartel_df = pd.concat([cartel_1, cartel_2])
cartel_df.shape

(767, 10)

In [19]:
cartel_windows = make_windows(
    cartel_df,
    window=18,
    price_col="p",
    state_col="S",
    id_col="market_id",
    time_col="t",
)

In [20]:
cartel_windows.head()

,market_id,window_start,window_end,window_length,Price 1,Price 2,Price 3,Price 4,Price 5,Price 6,...,Price 14,Price 15,Price 16,Price 17,Price 18,share_C,share_T,share_K,state_mode,is_pure_80
0,beta_carotene,0,17,18,4.248495,4.254543,4.259975,4.265954,4.271707,4.277617,...,4.325384,4.332462,4.340241,4.347712,4.355373,0.0,0.0,1.0,2,1.0
1,beta_carotene,1,18,18,4.254543,4.259975,4.265954,4.271707,4.277617,4.283303,...,4.332462,4.340241,4.347712,4.355373,4.362732,0.0,0.0,1.0,2,1.0
2,beta_carotene,2,19,18,4.259975,4.265954,4.271707,4.277617,4.283303,4.289145,...,4.340241,4.347712,4.355373,4.362732,4.370280,0.0,0.0,1.0,2,1.0
3,beta_carotene,3,20,18,4.265954,4.271707,4.277617,4.283303,4.289145,4.294953,...,4.347712,4.355373,4.362732,4.370280,4.377771,0.0,0.0,1.0,2,1.0
4,beta_carotene,4,21,18,4.271707,4.277617,4.283303,4.289145,4.294953,4.300542,...,4.355373,4.362732,4.370280,4.377771,4.384968,0.0,0.0,1.0,2,1.0


In [21]:
cartel_windows["cartel_share"] = cartel_windows["share_K"]
cartel_windows["cartel_window"] = (cartel_windows["share_K"] >= 0.5).astype(int)
cartel_windows["pure_cartel_window"] = (cartel_windows["share_K"] >= 0.8).astype(int)

In [22]:
cartel_windows.head()

,market_id,window_start,window_end,window_length,Price 1,Price 2,Price 3,Price 4,Price 5,Price 6,...,Price 17,Price 18,share_C,share_T,share_K,state_mode,is_pure_80,cartel_share,cartel_window,pure_cartel_window
0,beta_carotene,0,17,18,4.248495,4.254543,4.259975,4.265954,4.271707,4.277617,...,4.347712,4.355373,0.0,0.0,1.0,2,1.0,1.0,1,1
1,beta_carotene,1,18,18,4.254543,4.259975,4.265954,4.271707,4.277617,4.283303,...,4.355373,4.362732,0.0,0.0,1.0,2,1.0,1.0,1,1
2,beta_carotene,2,19,18,4.259975,4.265954,4.271707,4.277617,4.283303,4.289145,...,4.362732,4.370280,0.0,0.0,1.0,2,1.0,1.0,1,1
3,beta_carotene,3,20,18,4.265954,4.271707,4.277617,4.283303,4.289145,4.294953,...,4.370280,4.377771,0.0,0.0,1.0,2,1.0,1.0,1,1
4,beta_carotene,4,21,18,4.271707,4.277617,4.283303,4.289145,4.294953,4.300542,...,4.377771,4.384968,0.0,0.0,1.0,2,1.0,1.0,1,1


In [23]:
cartel_windows["cartel_window"].value_counts()


cartel_window
1    482
0    183
Name: count, dtype: int64

In [25]:
feature_df = feature_eng_syn(cartel_windows)
feature_df.head()

,market_id,window_start,window_end,window_length,Price 1,Price 2,Price 3,Price 4,Price 5,Price 6,...,CoV_change,zero_change_fraction,AR_1,AR_2,kurtosis_change,max_abs_ret,pos_vol,neg_vol,level_vol,price_range
0,beta_carotene,0,17,18,4.248495,4.254543,4.259975,4.265954,4.271707,4.277617,...,0.136719,0.0,0.698153,0.661551,-0.814392,0.007896,0.000886,NaN,0.032839,0.106878
1,beta_carotene,1,18,18,4.254543,4.259975,4.265954,4.271707,4.277617,4.283303,...,0.140289,0.0,0.735039,0.692594,-1.417541,0.007896,0.000920,NaN,0.033525,0.108189
2,beta_carotene,2,19,18,4.259975,4.265954,4.271707,4.277617,4.283303,4.289145,...,0.138954,0.0,0.753167,0.707695,-1.826771,0.007896,0.000929,NaN,0.034287,0.110305
3,beta_carotene,3,20,18,4.265954,4.271707,4.277617,4.283303,4.289145,4.294953,...,0.140075,0.0,0.765584,0.721161,-2.046572,0.007896,0.000950,NaN,0.035040,0.111817
4,beta_carotene,4,21,18,4.271707,4.277617,4.283303,4.289145,4.294953,4.300542,...,0.136265,0.0,0.759015,0.707433,-2.044091,0.007896,0.000936,NaN,0.035787,0.113260


In [27]:
path = Path("../data/cartel_data/feature")
path.mkdir(parents=True, exist_ok=True)

In [31]:
score = pd.read_parquet("../data/cartel_data/score/cartel_scoring_L18.parquet")
score.head()

,market_id,window_start,window_end,window_length,Price 1,Price 2,Price 3,Price 4,Price 5,Price 6,...,recon_outside99,score_outside95,score_outside99,cartel_and_in_manifold,high_score_and_cartel,recon_sqerr_volatility,recon_sqerr_zero_change_fraction,recon_sqerr_max_abs_ret,recon_sqerr_AR_1,recon_sqerr_price_range
0,beta_carotene,0,17,18,4.248495,4.254543,4.259975,4.265954,4.271707,4.277617,...,False,False,False,True,False,0.083051,0.036805,0.008704,0.002549,0.160836
1,beta_carotene,1,18,18,4.254543,4.259975,4.265954,4.271707,4.277617,4.283303,...,False,False,False,True,False,0.079254,0.036591,0.008560,0.002079,0.151108
2,beta_carotene,2,19,18,4.259975,4.265954,4.271707,4.277617,4.283303,4.289145,...,False,False,False,True,False,0.080748,0.036046,0.009392,0.002065,0.156367
3,beta_carotene,3,20,18,4.265954,4.271707,4.277617,4.283303,4.289145,4.294953,...,False,False,False,True,False,0.081586,0.035623,0.010099,0.002054,0.160174
4,beta_carotene,4,21,18,4.271707,4.277617,4.283303,4.289145,4.294953,4.300542,...,False,False,False,True,False,0.086154,0.035163,0.011257,0.002376,0.173824


In [38]:
score.columns

Index(['market_id', 'window_start', 'window_end', 'window_length', 'Price 1',
       'Price 2', 'Price 3', 'Price 4', 'Price 5', 'Price 6', 'Price 7',
       'Price 8', 'Price 9', 'Price 10', 'Price 11', 'Price 12', 'Price 13',
       'Price 14', 'Price 15', 'Price 16', 'Price 17', 'Price 18', 'share_C',
       'share_T', 'share_K', 'state_mode', 'is_pure_80', 'cartel_share',
       'cartel_window', 'pure_cartel_window', 'mean_change', 'volatility',
       'CoV_change', 'zero_change_fraction', 'AR_1', 'AR_2', 'kurtosis_change',
       'max_abs_ret', 'pos_vol', 'neg_vol', 'level_vol', 'price_range', 'z1',
       'z2', 'recon_error', 'conduct_score_centered', 'recon_outside95',
       'recon_outside99', 'score_outside95', 'score_outside99',
       'cartel_and_in_manifold', 'high_score_and_cartel',
       'recon_sqerr_volatility', 'recon_sqerr_zero_change_fraction',
       'recon_sqerr_max_abs_ret', 'recon_sqerr_AR_1',
       'recon_sqerr_price_range', 'score_rank'],
      dtype='str')

In [32]:
score["pure_cartel_window"] = (score["share_K"] >= 0.8).astype(int)

score.groupby("pure_cartel_window")["conduct_score_centered"].mean()

pure_cartel_window
0    1.033065
1    1.683096
Name: conduct_score_centered, dtype: float32

In [58]:
tau95 = 2.268063545227051

In [62]:
import plotly.express as px

df = score.sort_values(["market_id", "window_start"])

fig = px.line(
    df,
    x="window_start",
    y="conduct_score_centered",
    facet_row="market_id",
    height=900,
    title="Conduct Score Over Time Across Known Cartels"
)

# remove repeated y-axis titles
for axis in fig.layout:
    if axis.startswith("yaxis"):
        fig.layout[axis].title.text = ""

# add one global y-axis label
fig.add_annotation(
    x=-0.05,
    y=0.5,
    xref="paper",
    yref="paper",
    text="Centered Conduct Score",
    showarrow=False,
    textangle=-90,
    font=dict(size=14)
)

fig.show()

In [46]:
fig = px.scatter(
    df,
    x="share_K",
    y="conduct_score_centered",
    color="market_id",
    title="Conduct Score vs Cartel Share"
)

fig.show()

In [47]:
fig = px.box(
    df,
    x="cartel_window",
    y="conduct_score_centered",
    color="market_id",
    title="Conduct Score Distribution in Cartel vs Non-Cartel Windows"
)

fig.show()

In [48]:
df.groupby("market_id").agg(
    mean_score=("conduct_score_centered", "mean"),
    pct_above_tau95=("score_outside95", "mean"),
    mean_cartel_share=("share_K", "mean")
)

,mean_score,pct_above_tau95,mean_cartel_share
market_id,,,
beta_carotene,2.893241,0.204724,0.799213
citric_acid,0.861967,0.000000,0.391304
lysine,0.426413,0.000000,0.626984
vitamin_A,2.576974,0.204724,0.799213
vitamin_C,0.702429,0.000000,0.799213
vitamin_E,0.551890,0.000000,0.799213


In [49]:
df.groupby("market_id")["share_K"].describe()

,count,mean,std,min,25%,50%,75%,max
market_id,,,,,,,,
beta_carotene,127.0,0.799213,0.371496,0.0,0.861111,1.000000,1.000000,1.0
citric_acid,115.0,0.391304,0.433365,0.0,0.000000,0.166667,0.944444,1.0
lysine,42.0,0.626984,0.380827,0.0,0.291667,0.750000,1.000000,1.0
vitamin_A,127.0,0.799213,0.371496,0.0,0.861111,1.000000,1.000000,1.0
vitamin_C,127.0,0.799213,0.371496,0.0,0.861111,1.000000,1.000000,1.0
vitamin_E,127.0,0.799213,0.371496,0.0,0.861111,1.000000,1.000000,1.0


In [52]:
fig = px.scatter(
    df,
    x="recon_error",
    y="conduct_score_centered",
    color="market_id",
    title="Conduct Score vs Cartel Share"
)


fig.show()

In [54]:
df["score_rank"] = df["conduct_score_centered"].rank(pct=True)

In [56]:
fig = px.box(
    df,
    x="cartel_window",
    y="score_rank",
    color="market_id",
    title="Score Rank Distribution in Cartel vs Non-Cartel Windows"
)

fig.show()

In [60]:
px.scatter(
    df,
    x="share_K",
    y="volatility",
    color="market_id"
)

In [61]:
px.scatter(
    df,
    x="z1",
    y="z2",
    color="market_id"
)

In [63]:
feat_cols = ["volatility", "zero_change_fraction", "max_abs_ret", "AR_1", "price_range"]
summary = []
for m, g in score.groupby("market_id"):
    g0 = g[g["cartel_window"] == 0]
    g1 = g[g["cartel_window"] == 1]
    row = {"market_id": m}
    for c in feat_cols:
        row[f"{c}_diff"] = g1[c].mean() - g0[c].mean()
    summary.append(row)

feature_diff = pd.DataFrame(summary)
feature_diff

,market_id,volatility_diff,zero_change_fraction_diff,max_abs_ret_diff,AR_1_diff,price_range_diff
0,beta_carotene,0.000811,0.056125,0.004500,0.184360,0.054332
1,citric_acid,-0.002181,0.000000,-0.001340,0.296123,-0.005733
2,lysine,0.004220,0.000000,0.010265,0.023138,0.062011
3,vitamin_A,-0.006016,0.117647,-0.011858,0.346737,-0.060037
4,vitamin_C,-0.000441,0.000000,0.000750,0.339690,0.019288
5,vitamin_E,-0.008507,0.000000,-0.025554,0.078106,-0.196007
